# 05 - Clustering de Selecciones por Estilo de Juego (K-Means)

**Objetivo**: agrupar las selecciones nacionales en clusters según su estilo histórico (ofensivo / defensivo / equilibrado / etc.) usando K-Means sobre features agregadas.

**Uso esperado**:
1. **Narrativa en la UI premium**: cuando el usuario consulta un partido, mostrar el estilo de cada equipo (ej. "Brasil: estilo ofensivo, alta posesión").
2. **Feature adicional para los modelos supervisados**: el cluster del equipo puede ser una feature categórica más en el clasificador W/D/L y los regresores.
3. **Insight académico**: mostrar que el clustering no supervisado puede descubrir patrones interpretables.

**Dataset**: usamos Martj42 + WC2022 para construir un perfil agregado por selección:
- **Goles a favor / en contra promedio** (de los últimos 8 años) → Martj42
- **Win rate, draw rate** (últimos 8 años) → Martj42
- **Posesión, corners, intentos, faltas** (promedios WC2022) → solo selecciones que jugaron en Qatar 2022

**Algoritmo**: K-Means con K seleccionado por el método del codo y silhouette score.


## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0]  # ml/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

from src.data import load_international_results, load_wc2022_matches

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)
RANDOM_STATE = 42


## 1. Construir perfil agregado por selección

**Features por equipo** (calculadas sobre partidos 2018-2026):
- `avg_gf`, `avg_ga`, `avg_gd`: goles a favor / contra / diferencial
- `win_rate`, `draw_rate`, `loss_rate`
- `n_matches`: número de partidos (peso/confiabilidad)
- `pct_high_scoring`: % de partidos con >2.5 goles

**Filtrado**: solo selecciones con al menos 20 partidos en el periodo (descartar selecciones marginales sin suficiente data).


In [ ]:
matches = load_international_results()
recent = matches[matches['date'] >= '2018-01-01'].copy()
recent = recent.dropna(subset=['home_score', 'away_score'])

# Convertir a formato long (1 fila por equipo por partido)
home = recent.rename(columns={'home_team': 'team', 'away_team': 'opponent',
                              'home_score': 'gf', 'away_score': 'ga'}).assign(is_home=1)
away = recent.rename(columns={'away_team': 'team', 'home_team': 'opponent',
                              'away_score': 'gf', 'home_score': 'ga'}).assign(is_home=0)
long = pd.concat([home[['date','team','opponent','gf','ga','is_home']],
                  away[['date','team','opponent','gf','ga','is_home']]], ignore_index=True)
long['gd'] = long['gf'] - long['ga']
long['win'] = (long['gd'] > 0).astype(int)
long['draw'] = (long['gd'] == 0).astype(int)
long['loss'] = (long['gd'] < 0).astype(int)
long['high_scoring'] = ((long['gf'] + long['ga']) > 2.5).astype(int)

print(f'Partidos (long): {len(long):,}')
print(f'Equipos únicos: {long.team.nunique()}')


In [ ]:
# Agregados por equipo
team_profile = long.groupby('team').agg(
    avg_gf=('gf', 'mean'),
    avg_ga=('ga', 'mean'),
    avg_gd=('gd', 'mean'),
    win_rate=('win', 'mean'),
    draw_rate=('draw', 'mean'),
    loss_rate=('loss', 'mean'),
    pct_high_scoring=('high_scoring', 'mean'),
    n_matches=('gf', 'count'),
).round(3).reset_index()

# Filtrar a selecciones con suficiente historial reciente
MIN_MATCHES = 20
team_profile = team_profile[team_profile['n_matches'] >= MIN_MATCHES].reset_index(drop=True)
print(f'Selecciones con >= {MIN_MATCHES} partidos: {len(team_profile)}')
team_profile.sort_values('win_rate', ascending=False).head(15)


### 1.1 Enriquecer con stats de WC2022 (cuando disponible)

Para las 32 selecciones que jugaron en Qatar 2022, agregamos posesión, corners y faltas promedio.


In [ ]:
wc = load_wc2022_matches()
rows = []
for _, r in wc.iterrows():
    for side in (1, 2):
        rows.append({
            'team': r[f'team{side}'].title(),  # WC2022 está en MAYÚSCULAS, normalizar
            'possession': r[f'possession_team{side}'],
            'corners': r[f'corners_team{side}'],
            'fouls': r[f'fouls_against_team{side}'],
            'attempts': r[f'total_attempts_team{side}'],
            'on_target': r[f'on_target_attempts_team{side}'],
        })
wc_team_avg = pd.DataFrame(rows).groupby('team').mean().reset_index()
print(f'Selecciones WC2022 con stats avanzadas: {len(wc_team_avg)}')

# Merge case-insensitive
team_profile['team_lower'] = team_profile['team'].str.lower()
wc_team_avg['team_lower'] = wc_team_avg['team'].str.lower()
enriched = team_profile.merge(
    wc_team_avg.drop(columns=['team']),
    on='team_lower', how='left'
).drop(columns=['team_lower'])

print(f'Selecciones con stats WC2022: {enriched["possession"].notna().sum()}')
enriched.head()


## 2. Selección de K óptimo (método del codo + silhouette)

Para el clustering principal usamos solo las features básicas de Martj42 (disponibles para todos los equipos). Las stats WC2022 las podemos usar como información adicional para describir los clusters.


In [ ]:
# Features para clustering
BASIC_FEATURES = ['avg_gf', 'avg_ga', 'avg_gd', 'win_rate', 'draw_rate', 'pct_high_scoring']
X = enriched[BASIC_FEATURES].values

# Estandarizar (K-Means es sensible a la escala)
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

# Método del codo
k_range = range(2, 11)
inertias = []
silhouettes = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(Xs)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(Xs, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(list(k_range), inertias, marker='o', color='#2E86AB')
axes[0].set_title('Método del codo (inercia)')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inercia')
axes[0].grid(True)

axes[1].plot(list(k_range), silhouettes, marker='o', color='#A23B72')
axes[1].set_title('Silhouette score')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
axes[1].grid(True)
best_k = k_range[int(np.argmax(silhouettes))]
axes[1].axvline(best_k, color='red', linestyle='--', label=f'best K={best_k}')
axes[1].legend()
plt.tight_layout(); plt.show()

print(f'\nK óptimo por silhouette: {best_k}')
print(f'Silhouette scores: {dict(zip(k_range, [round(s, 3) for s in silhouettes]))}')


## 3. Aplicar K-Means con el K óptimo

In [ ]:
# Forzar K=4 si el óptimo silhouette es demasiado bajo (p.ej. 2 clusters muy grandes)
# Para narrativa en UI, 3-5 clusters son útiles. Si el silhouette óptimo es K=2,
# usamos K=4 como compromiso interpretabilidad/calidad.
K = best_k if best_k >= 3 else 4
print(f'Usando K={K}')

kmeans = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20)
enriched['cluster'] = kmeans.fit_predict(Xs)

# Caracterizar cada cluster por sus promedios
cluster_profile = enriched.groupby('cluster')[BASIC_FEATURES + ['n_matches']].mean().round(3)
cluster_profile['size'] = enriched.groupby('cluster').size()
print('\nPerfil de cada cluster:')
print(cluster_profile)


In [ ]:
# Etiquetar cada cluster con un nombre humano basado en su perfil
def label_cluster(row):
    wr, gd, hs = row['win_rate'], row['avg_gd'], row['pct_high_scoring']
    if gd > 0.5 and wr > 0.5:
        return 'élite ofensivo'
    elif gd > 0 and wr > 0.4:
        return 'equilibrado fuerte'
    elif abs(gd) <= 0.5 and 0.3 <= wr <= 0.5:
        return 'equilibrado medio'
    elif gd < -0.5 or wr < 0.25:
        return 'defensivo / débil'
    elif hs > 0.55:
        return 'goleador inestable'
    else:
        return 'mixto'

cluster_profile['label'] = cluster_profile.apply(label_cluster, axis=1)
label_map = cluster_profile['label'].to_dict()
enriched['cluster_label'] = enriched['cluster'].map(label_map)

print('Etiquetas asignadas a cada cluster:')
print(cluster_profile[['win_rate', 'avg_gd', 'pct_high_scoring', 'size', 'label']])


In [ ]:
# Algunos equipos representativos por cluster
famous = ['Brazil', 'Argentina', 'France', 'Germany', 'Spain', 'England', 'Portugal',
          'Netherlands', 'Italy', 'Belgium', 'Colombia', 'Mexico', 'United States',
          'Uruguay', 'Croatia', 'Japan', 'South Korea', 'Australia', 'Morocco',
          'Senegal', 'Ghana', 'Saudi Arabia', 'Qatar', 'Canada']
famous_in = enriched[enriched['team'].isin(famous)][
    ['team', 'win_rate', 'avg_gf', 'avg_ga', 'avg_gd', 'cluster', 'cluster_label']
].sort_values(['cluster', 'win_rate'], ascending=[True, False])
print('Asignación de equipos conocidos a clusters:')
print(famous_in.to_string(index=False))


## 4. Visualización en 2D con PCA

In [ ]:
# Reducir a 2 componentes principales para visualizar
pca = PCA(n_components=2, random_state=RANDOM_STATE)
Xp = pca.fit_transform(Xs)
enriched['pca1'] = Xp[:, 0]
enriched['pca2'] = Xp[:, 1]

fig, ax = plt.subplots(figsize=(12, 8))
for cluster_id in sorted(enriched['cluster'].unique()):
    sub = enriched[enriched['cluster'] == cluster_id]
    label = sub['cluster_label'].iloc[0]
    ax.scatter(sub['pca1'], sub['pca2'], s=80, alpha=0.7, label=f'Cluster {cluster_id}: {label}')

# Anotar algunos equipos famosos
for _, r in enriched[enriched['team'].isin(famous)].iterrows():
    ax.annotate(r['team'], (r['pca1'], r['pca2']), fontsize=8, alpha=0.85,
                xytext=(4, 4), textcoords='offset points')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var.)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var.)')
ax.set_title(f'K-Means clusters de selecciones (K={K})')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 5. Cross-check con stats WC2022

In [ ]:
# Para las selecciones con stats WC2022, ¿los clusters reflejan diferencias en posesión y corners?
wc_clusters = enriched[enriched['possession'].notna()]
print(f'Selecciones con datos WC2022: {len(wc_clusters)}')

wc_by_cluster = wc_clusters.groupby('cluster_label')[
    ['possession', 'corners', 'attempts', 'on_target', 'fouls']
].mean().round(2)
print('\nPromedios WC2022 por cluster:')
print(wc_by_cluster)


## 6. Persistir clusters y modelo

Guardamos el modelo KMeans y la tabla `{team -> cluster_label}` para uso del backend.


In [ ]:
from pathlib import Path

MODELS_DIR = ROOT / 'trained_models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump({
    'model': kmeans,
    'scaler': scaler,
    'features': BASIC_FEATURES,
    'cluster_labels': label_map,
    'team_to_cluster': dict(zip(enriched['team'], enriched['cluster'])),
    'team_to_label': dict(zip(enriched['team'], enriched['cluster_label'])),
    'k': K,
    'silhouette': float(silhouettes[best_k - 2]),
}, MODELS_DIR / 'clustering_teams.pkl')

# También guardamos la tabla de perfiles en CSV para la UI / backend
processed = ROOT / 'data' / 'processed'
processed.mkdir(parents=True, exist_ok=True)
enriched.to_csv(processed / 'team_profiles.csv', index=False)

print(f'✓ clustering_teams.pkl guardado en {MODELS_DIR}')
print(f'✓ team_profiles.csv guardado en {processed}')
print(f'\n{len(enriched)} selecciones perfiladas, {K} clusters identificados')


## 7. Próximos pasos

Con todos los modelos entrenados, pasamos a la fase de **despliegue**:

1. **Backend FastAPI** (`backend/`):
   - Cargar `classifier.pkl`, `regressor_total_goals.pkl`, etc. al arrancar la app.
   - Endpoints:
     - `GET /teams` → lista de selecciones del Mundial 2026
     - `POST /predict/result` → probabilidades W/D/L (gratuita)
     - `POST /predict/stats` → goles 1T/2T, tarjetas, corners (premium, requiere JWT)
     - `GET /teams/{team}/profile` → cluster + stats agregados
   - Autenticación JWT para la capa premium.
2. **Frontend React + Tailwind** (`frontend/`):
   - Página principal: selector de 2 equipos.
   - Página resultado: visualización con Chart.js de las probabilidades.
   - Página premium: dashboard de stats detalladas.
3. **Despliegue**: Docker Compose + Render/Railway para el backend, Vercel para el frontend.
